In [223]:
import math
import numpy as np
import random

In [224]:
class Value:
    def __init__(self, data, _children = (), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self._backward = lambda: None
        self.grad = 0

    def __repr__(self):
        return f"Value(data={self.data}, label={self.label}, gradient={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, _children = (self, other), _op='+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, _children = (self, other), _op='*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out;

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data**other.data, _children = (self, ), _op = f'**{other.data}')
        def _backward():
            d = other.data * (self.data ** (other.data - 1))
            self.grad += d * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return -1 * self

    def __sub__(self, other):
        return self + (-other)

    def __truediv__(self, other):
        return self * (other ** -1)


    def tanh(self):
        _dataoutput = (math.exp(2 * self.data) - 1)/(math.exp(2*self.data) + 1)
        out = Value(_dataoutput, _children= (self, ), _op = 'tanh')
        def _backward():
            t = 1 - (_dataoutput)**2
            self.grad += t * out.grad
        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        visited = set()
        res = []
        def build_topo(node):
            visited.add(node)  
            for child in node._prev:
                if child not in visited:
                    build_topo(child)
            res.append(node)         

        build_topo(self)
        for v in reversed(res):
            v._backward()



In [225]:
a = Value(2, label = 'a')
b = Value(-3, label = 'b')
c = Value(10, label = 'c')
f = Value(-2, label = 'f')
e = a * b
e.label = 'e'
d = e + c
d.label = 'd'
L = d * f
L.label = 'L'
L.backward()


In [226]:
a = Value(3.0)
b = a + a
b.backward()

In [227]:
x1 = Value(2, label = 'x1')
x2 = Value(0, label = 'x2')
w1 = Value(-3, label = 'w1')
w2 = Value(1, label = 'w2')
b = Value(6.8813735870195432, label = 'b')
x1w1 = x1 * w1
x1w1.label = 'x1*w1'
x2w2 = x2 * w2
x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = 'x1*w1 + x2*w2'
n =  x1w1 + x2w2 + b
n.label = 'x1*w1 + x2*w2 + b'
o = n.tanh()
o.backward()



In [228]:
a = Value(2, label = 'a')
b = Value(-3, label = 'b')
y = (a-b)/(a**2)
y.label = 'y'
y.backward()


In [229]:
a.data += 0.0001
y = (a-b)/(a**2)
y.backward()

In [230]:
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, vals):
        dot_products = [v*w for v,w in zip(vals, self.w)]
        res = sum(dot_products, self.b)
        return res.tanh()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, vals):
        outputs = []
        for n in self.neurons:
            outputs.append(n(vals))
        return outputs

    def parameters(self):
        params = []
        for n in self.neurons:
            params += n.parameters();
        return params
            

        

class MLP:
    def __init__(self, nin, nout):
        self.mlp = [nin] + nout
        self.layers = [Layer(self.mlp[i], self.mlp[i+1]) for i in range(len(self.mlp)-1)]
    def __call__(self, inputs):
        for layer in self.layers:
            new_inputs = layer(inputs)
            inputs = new_inputs
        return inputs[0] if len(inputs)==1 else inputs
    def parameters(self):
        params = []
        for layer in self.layers:
            params+= layer.parameters()
        return params

    


In [238]:
xs = [[3.0, -1.0, 2.0], [5.0, 2.0, 3.0], [-1.0, -2.0, -3.0], [6.9, 3.2, 0.0]]
ys = [1, -1, -1, 1]
n = MLP(3, [4, 4, 1])
for k in range(50):
    yout = [n(x) for x in xs];
    loss = sum([(y2-y1) ** 2 for y2,y1 in zip(yout, ys)])
    for p in n.parameters():
        p.grad = 0
    loss.backward()
    for p in n.parameters():
        p.data -= 0.05 * p.grad
    print(k, loss.data)
yout = [n(x) for x in xs];
yout





0 7.81981794106154
1 5.610253004242477
2 4.377474402523644
3 2.950074690681165
4 2.816892736088647
5 2.772575369486303
6 2.748928571530537
7 2.7336663522720106
8 2.722651187509143
9 2.7140248641296254
10 2.7067904839985553
11 2.7003256992143116
12 2.6941689424163275
13 2.6878946497055183
14 2.68100033085984
15 2.672738196977091
16 2.6617577513626243
17 2.6451525663962316
18 2.6153245536706895
19 2.546746265800333
20 2.3280984284305157
21 1.8188260466551993
22 1.4862615812684385
23 1.1951107150360927
24 0.9558061147550079
25 0.8343908296868483
26 1.1719255021341144
27 4.465181109885746
28 2.4121369886151087
29 2.314227701718413
30 1.2869275227594426
31 0.5793415993670721
32 0.4015879998884352
33 0.3257365211927503
34 0.27719480930565565
35 0.24122647571461187
36 0.2130566033795503
37 0.19034901560784281
38 0.17167916152330745
39 0.156089437037087
40 0.14290172306617094
41 0.13162012590714067
42 0.12187356131833474
43 0.11337919528508414
44 0.10591806528014583
45 0.09931831131629304
46 0

[Value(data=0.8758176085456029, label=, gradient=0),
 Value(data=-0.8412686852145291, label=, gradient=0),
 Value(data=-0.859908702699721, label=, gradient=0),
 Value(data=0.8773698241952355, label=, gradient=0)]